# Hyperparameter sweep for A_baseline

Companion to notebook 13 (`13-hp-sweep.ipynb`, which tunes
`D_dilated_reg`). The architecture sweep in notebook 12 picked the
winner purely by MotionSense macro-F1, ignoring its own stated
in-the-wild criterion. By that second criterion, **A_baseline**
actually beats every dilated variant on the Android sessions
(win_acc 0.7542 vs 0.6750-0.7167) while losing only ~1.5 pp on
MotionSense. This notebook gives A_baseline the *same* HP search
budget as D got in notebook 13, so the comparison is apples-to-apples
rather than a head start for the regularised variant.

**Selection protocol** (identical to notebook 13):
- All configurations are trained on subjects 0-14 (train) and
  early-stopped against subjects 15-18 (val).
- Validation macro-F1 is the **only** selection criterion.
- The held-out test set (subjects 19-23) is touched **once**, after
  the top-3 by val are selected.
- 5-fold subject-wise CV runs on the overall winner for the final
  confidence interval.
- In-the-wild robustness is reported, not used for selection (so the
  comparison with notebook 13 stays on the same selection criterion).

**Search space.** A_baseline as defined in notebook 12 has no L2,
no SpatialDropout1D and no label smoothing. C0 reproduces that
vanilla config; C1-C4 vary lr/bs; C5-C8 explore *adding* L2 and
SpatialDropout1D (knobs D already had); C9-C11 vary dense dropout
and label smoothing; C12-C13 combine the most promising one-factor
moves. 14 configurations total, matching notebook 13's budget so
compute time is comparable.

Architectural references:

> Wang, Z., Yan, W., & Oates, T. (2017). Time series classification from scratch with deep neural networks: A strong baseline. *IJCNN 2017*. https://doi.org/10.1109/IJCNN.2017.7966039

> Srivastava, N. et al. (2014). Dropout: A simple way to prevent neural networks from overfitting. *JMLR*, 15.

> Tompson, J. et al. (2015). Efficient object localization using convolutional networks. *CVPR 2015* (SpatialDropout). https://arxiv.org/abs/1411.4280

> Mueller, R., Kornblith, S., & Hinton, G. (2019). When does label smoothing help? *NeurIPS 2019*. https://arxiv.org/abs/1906.02629

In [1]:
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import tensorflow as tf, keras
from keras import layers, callbacks, regularizers, losses
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold

warnings.filterwarnings('ignore')
SEED = 42; np.random.seed(SEED); tf.random.set_seed(SEED)

_ROOT = Path('..').resolve(); sys.path.insert(0, str(_ROOT))
from utils.orientation_invariant_features import (
    compute_walking_frame_features_v2, WALKING_FRAME_V2_COLS,
)

## Data loading (same as previous notebooks)

In [2]:
ACT_LABELS = ['dws','ups','wlk','jog','std','sit']
TRIAL_CODES = {
    'dws': [1,2,11], 'ups': [3,4,12], 'wlk': [7,8,15],
    'jog': [9,16],   'std': [6,14],   'sit': [5,13],
}


def get_ds_infos(): return pd.read_csv('../../data/data_subjects_info.csv')
def set_data_types(types):
    return [[t+'.x',t+'.y',t+'.z'] if t!='attitude' else ['attitude.roll','attitude.pitch','attitude.yaw'] for t in types]


def create_time_series(dt_list, act_labels, trial_codes):
    n_cols = len(dt_list)*3
    dataset = np.zeros((0, n_cols+7))
    ds_list = get_ds_infos()
    for sub_id in ds_list['code']:
        for act_id, act in enumerate(act_labels):
            for trial in trial_codes[act]:
                f = f'../../data/A_DeviceMotion_data/{act}_{trial}/sub_{int(sub_id)}.csv'
                raw = pd.read_csv(f).drop(['Unnamed: 0'], axis=1)
                vals = np.zeros((len(raw), n_cols))
                for x, axes in enumerate(dt_list):
                    vals[:, x*3:(x+1)*3] = raw[axes].values
                lbls = np.array([[act_id, sub_id-1,
                                  ds_list['weight'][sub_id-1],
                                  ds_list['height'][sub_id-1],
                                  ds_list['age'][sub_id-1],
                                  ds_list['gender'][sub_id-1], trial]] * len(raw))
                dataset = np.append(dataset, np.concatenate((vals, lbls), axis=1), axis=0)
    cols = [c for axes in dt_list for c in axes] + ['act','id','weight','height','age','gender','trial']
    return pd.DataFrame(data=dataset, columns=cols)


def sliding_windows(data, feature_cols, w=128, s=64):
    X, y, g = [], [], []
    for (sid, act, _), b in data.groupby(['id','act','trial'], sort=False):
        v = b[feature_cols].to_numpy()
        for st in range(0, len(v)-w+1, s):
            X.append(v[st:st+w]); y.append(act); g.append(sid)
    return np.array(X), np.array(y), np.array(g)


dt_list = set_data_types(['attitude','gravity','rotationRate','userAcceleration'])
dataset = create_time_series(dt_list, ACT_LABELS, TRIAL_CODES)
for col in ('act','id','trial'): dataset[col] = dataset[col].astype(int)
features_df = compute_walking_frame_features_v2(dataset, fs_hz=50.0, smooth_seconds=5.0)

train_ids = list(range(0, 15)); val_ids = list(range(15, 19)); test_ids = list(range(19, 24))
X_train, y_train, g_train = sliding_windows(features_df[features_df['id'].isin(train_ids)], WALKING_FRAME_V2_COLS)
X_val,   y_val,   g_val   = sliding_windows(features_df[features_df['id'].isin(val_ids)],   WALKING_FRAME_V2_COLS)
X_test,  y_test,  g_test  = sliding_windows(features_df[features_df['id'].isin(test_ids)],  WALKING_FRAME_V2_COLS)
y_train, y_val, y_test = y_train.astype(int), y_val.astype(int), y_test.astype(int)
N_CHAN = len(WALKING_FRAME_V2_COLS)


def normalize_dyn(X, eps=1e-8):
    out = X.copy().astype(np.float32)
    return (out - out.mean(axis=1, keepdims=True)) / (out.std(axis=1, keepdims=True) + eps)


X_train_n = normalize_dyn(X_train); X_val_n = normalize_dyn(X_val); X_test_n = normalize_dyn(X_test)
cw = compute_class_weight('balanced', classes=np.arange(6), y=y_train)
CLASS_WEIGHT = {int(i): float(w) for i, w in enumerate(cw)}
print(f'train {X_train_n.shape}, val {X_val_n.shape}, test {X_test_n.shape}')

train (13282, 128, 8), val (3905, 128, 8), test (4352, 128, 8)


## Architecture builder (parameterised A_baseline)

Same three-block Conv1D + MaxPool stack as notebook 11/12's
`build_cnn_baseline` / `build_A_baseline`, but with optional L2,
SpatialDropout1D and label smoothing (label smoothing is applied
in the loss, not the builder). When `l2=0`, `spatial_dropout=0`,
`dense_dropout=0.3` and `label_smoothing=0` we reproduce the exact
notebook 12 A_baseline.

In [3]:
def build_A_baseline_param(input_shape=(128, N_CHAN), n_classes=6,
                            l2=0.0, spatial_dropout=0.0, dense_dropout=0.3):
    reg = regularizers.l2(l2) if l2 > 0 else None
    inp = keras.Input(shape=input_shape)
    x = layers.Conv1D(64, 5, activation='relu', padding='same', kernel_regularizer=reg)(inp)
    x = layers.BatchNormalization()(x)
    if spatial_dropout > 0: x = layers.SpatialDropout1D(spatial_dropout)(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 5, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    if spatial_dropout > 0: x = layers.SpatialDropout1D(spatial_dropout)(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=reg)(x)
    x = layers.Dropout(dense_dropout)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    return keras.Model(inp, out, name='cnn_A_baseline_param')

## Sweep configurations

C0 reproduces the notebook-12 vanilla A_baseline. C1-C4 sweep
learning rate and batch size. C5-C8 add the regularisers
(L2, SpatialDropout1D) that D_dilated_reg had baked in.
C9-C11 vary dense dropout and add label smoothing.
C12-C13 combine the most promising one-factor moves.

In [4]:
CONFIGS = [
    # name,           lr,    bs,  l2,    sd,   dd,   ls
    ('C0_vanilla',    1e-3,  32,  0.0,   0.0,  0.3,  0.0),
    ('C1_low_lr',     5e-4,  32,  0.0,   0.0,  0.3,  0.0),
    ('C2_high_lr',    2e-3,  32,  0.0,   0.0,  0.3,  0.0),
    ('C3_small_bs',   1e-3,  16,  0.0,   0.0,  0.3,  0.0),
    ('C4_large_bs',   1e-3,  64,  0.0,   0.0,  0.3,  0.0),
    ('C5_l2_light',   1e-3,  32,  5e-5,  0.0,  0.3,  0.0),
    ('C6_l2_heavy',   1e-3,  32,  5e-4,  0.0,  0.3,  0.0),
    ('C7_sd_light',   1e-3,  32,  0.0,   0.1,  0.3,  0.0),
    ('C8_sd_heavy',   1e-3,  32,  0.0,   0.2,  0.3,  0.0),
    ('C9_less_dd',    1e-3,  32,  0.0,   0.0,  0.2,  0.0),
    ('C10_more_dd',   1e-3,  32,  0.0,   0.0,  0.4,  0.0),
    ('C11_ls',        1e-3,  32,  0.0,   0.0,  0.3,  0.05),
    ('C12_combo_A',   1e-3,  32,  1e-4,  0.2,  0.3,  0.05),
    ('C13_combo_B',   5e-4,  32,  1e-4,  0.1,  0.3,  0.05),
]
print(f'{len(CONFIGS)} configurations queued')

14 configurations queued


## Train + evaluate each configuration on val

In [5]:
def train_one(name, lr, bs, l2, sd, dd, ls):
    tf.keras.backend.clear_session(); tf.random.set_seed(SEED); np.random.seed(SEED)
    model = build_A_baseline_param(l2=l2, spatial_dropout=sd, dense_dropout=dd)
    if ls > 0:
        y_train_oh = tf.one_hot(y_train, depth=6).numpy()
        y_val_oh = tf.one_hot(y_val, depth=6).numpy()
        loss = losses.CategoricalCrossentropy(label_smoothing=ls)
        ytr, yva = y_train_oh, y_val_oh
    else:
        loss = losses.SparseCategoricalCrossentropy()
        ytr, yva = y_train, y_val
    model.compile(optimizer=keras.optimizers.Adam(lr), loss=loss, metrics=['accuracy'])
    cb = [callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
          callbacks.ReduceLROnPlateau(monitor='val_loss', patience=5, factor=0.5, min_lr=1e-6)]
    t0 = time.time()
    history = model.fit(X_train_n, ytr, validation_data=(X_val_n, yva),
                        epochs=50, batch_size=bs, class_weight=CLASS_WEIGHT,
                        callbacks=cb, verbose=0)
    elapsed = time.time() - t0
    yvp = model.predict(X_val_n, verbose=0).argmax(axis=1)
    val_f1 = f1_score(y_val, yvp, average='macro')
    return {
        'name': name, 'lr': lr, 'bs': bs, 'l2': l2, 'sd': sd, 'dd': dd, 'ls': ls,
        'val_f1': val_f1, 'epochs': len(history.history['loss']),
        'elapsed_s': elapsed, 'model': model,
    }


sweep_results = []
for cfg in CONFIGS:
    r = train_one(*cfg)
    sweep_results.append(r)
    print(f'{r["name"]:18s}  val_F1={r["val_f1"]:.4f}  epochs={r["epochs"]:>2d}  {r["elapsed_s"]:.0f}s')


C0_vanilla          val_F1=0.8250  epochs=12  69s
C1_low_lr           val_F1=0.7840  epochs=11  71s
C2_high_lr          val_F1=0.7880  epochs=11  69s
C3_small_bs         val_F1=0.8147  epochs=11  98s
C4_large_bs         val_F1=0.8039  epochs=14  62s
C5_l2_light         val_F1=0.8621  epochs=11  70s
C6_l2_heavy         val_F1=0.8014  epochs=15  95s
C7_sd_light         val_F1=0.8494  epochs=14  73s
C8_sd_heavy         val_F1=0.8327  epochs=14  85s
C9_less_dd          val_F1=0.7820  epochs=11  60s
C10_more_dd         val_F1=0.8396  epochs=11  62s
C11_ls              val_F1=0.8445  epochs=38  202s
C12_combo_A         val_F1=0.8624  epochs=41  262s
C13_combo_B         val_F1=0.8538  epochs=25  160s


## Selection - top 3 by val_F1

In [6]:
sweep_df = pd.DataFrame([{k: r[k] for k in ('name','lr','bs','l2','sd','dd','ls','val_f1','epochs','elapsed_s')}
                          for r in sweep_results])
sweep_df = sweep_df.sort_values('val_f1', ascending=False).reset_index(drop=True)
print(sweep_df.to_string())

os.makedirs('../results', exist_ok=True)
sweep_df.to_csv('../results/hp_sweep_baseline_val.csv', index=False)

           name      lr  bs       l2   sd   dd    ls    val_f1  epochs   elapsed_s
0   C12_combo_A  0.0010  32  0.00010  0.2  0.3  0.05  0.862435      41  262.479953
1   C5_l2_light  0.0010  32  0.00005  0.0  0.3  0.00  0.862149      11   70.001522
2   C13_combo_B  0.0005  32  0.00010  0.1  0.3  0.05  0.853808      25  159.910965
3   C7_sd_light  0.0010  32  0.00000  0.1  0.3  0.00  0.849411      14   73.153770
4        C11_ls  0.0010  32  0.00000  0.0  0.3  0.05  0.844473      38  202.498813
5   C10_more_dd  0.0010  32  0.00000  0.0  0.4  0.00  0.839645      11   61.564537
6   C8_sd_heavy  0.0010  32  0.00000  0.2  0.3  0.00  0.832675      14   85.152217
7    C0_vanilla  0.0010  32  0.00000  0.0  0.3  0.00  0.824954      12   68.987550
8   C3_small_bs  0.0010  16  0.00000  0.0  0.3  0.00  0.814715      11   98.190116
9   C4_large_bs  0.0010  64  0.00000  0.0  0.3  0.00  0.803931      14   62.044131
10  C6_l2_heavy  0.0010  32  0.00050  0.0  0.3  0.00  0.801352      15   94.529268
11  

## Evaluate top 3 on the held-out test set

In [7]:
top3 = sweep_df.head(3)['name'].tolist()
top3_models = {r['name']: r for r in sweep_results if r['name'] in top3}
print(f'Top 3 by val_F1: {top3}')

test_rows = []
for name in top3:
    r = top3_models[name]
    ytp = r['model'].predict(X_test_n, verbose=0).argmax(axis=1)
    test_f1 = f1_score(y_test, ytp, average='macro')
    per_class = f1_score(y_test, ytp, average=None)
    test_rows.append({
        'name': name,
        'val_f1': r['val_f1'],
        'test_f1': test_f1,
        **{f'F1_{a}': float(v) for a, v in zip(ACT_LABELS, per_class)},
    })

test_df = pd.DataFrame(test_rows).set_index('name')
print(test_df.round(4).to_string())
test_df.to_csv('../results/hp_sweep_baseline_test.csv')

winner_name = test_df['test_f1'].idxmax()
print(f'\nOverall winner: {winner_name}  test macro-F1 = {test_df.loc[winner_name, "test_f1"]:.4f}')

Top 3 by val_F1: ['C12_combo_A', 'C5_l2_light', 'C13_combo_B']
             val_f1  test_f1  F1_dws  F1_ups  F1_wlk  F1_jog  F1_std  F1_sit
name                                                                        
C12_combo_A  0.8624   0.9413  0.9057  0.9622  0.9382  0.9894  0.9179  0.9342
C5_l2_light  0.8621   0.9054  0.8883  0.8756  0.9180  0.9918  0.8683  0.8903
C13_combo_B  0.8538   0.9350  0.9089  0.9423  0.9403  0.9907  0.9083  0.9194

Overall winner: C12_combo_A  test macro-F1 = 0.9413


## 5-fold CV for the overall winner

In [8]:
winner_row = [r for r in sweep_results if r['name'] == winner_name][0]
lr_w, bs_w, l2_w, sd_w, dd_w, ls_w = (winner_row['lr'], winner_row['bs'], winner_row['l2'],
                                       winner_row['sd'], winner_row['dd'], winner_row['ls'])
print(f'CV config: lr={lr_w}, bs={bs_w}, l2={l2_w}, sd={sd_w}, dd={dd_w}, ls={ls_w}')

X_full, y_full, g_full = sliding_windows(features_df, WALKING_FRAME_V2_COLS)
y_full = y_full.astype(int); X_full_n = normalize_dyn(X_full)

gkf = GroupKFold(n_splits=5); fold_f1s = []
for fold, (tr, te) in enumerate(gkf.split(X_full_n, y_full, groups=g_full)):
    tf.keras.backend.clear_session(); tf.random.set_seed(SEED); np.random.seed(SEED)
    m = build_A_baseline_param(l2=l2_w, spatial_dropout=sd_w, dense_dropout=dd_w)
    if ls_w > 0:
        loss = losses.CategoricalCrossentropy(label_smoothing=ls_w)
        ytr = tf.one_hot(y_full[tr], depth=6).numpy()
    else:
        loss = losses.SparseCategoricalCrossentropy(); ytr = y_full[tr]
    m.compile(optimizer=keras.optimizers.Adam(lr_w), loss=loss, metrics=['accuracy'])
    cw_f = compute_class_weight('balanced', classes=np.arange(6), y=y_full[tr])
    m.fit(X_full_n[tr], ytr, epochs=25, batch_size=bs_w,
          class_weight={int(i): float(w) for i, w in enumerate(cw_f)}, verbose=0)
    yfp = m.predict(X_full_n[te], verbose=0).argmax(axis=1)
    f1 = f1_score(y_full[te], yfp, average='macro'); fold_f1s.append(f1)
    print(f'fold {fold+1}  subj={sorted(np.unique(g_full[te]).astype(int).tolist())}  macro-F1={f1:.4f}')
fold_f1s = np.array(fold_f1s)
print(f'\n{winner_name} 5-fold CV macro-F1: {fold_f1s.mean():.4f} ± {fold_f1s.std():.4f}')

CV config: lr=0.001, bs=32, l2=0.0001, sd=0.2, dd=0.3, ls=0.05
fold 1  subj=[9, 13, 18, 22]  macro-F1=0.8453
fold 2  subj=[7, 8, 19, 20, 23]  macro-F1=0.9125
fold 3  subj=[5, 6, 11, 15, 21]  macro-F1=0.9580
fold 4  subj=[0, 4, 14, 16, 17]  macro-F1=0.9144
fold 5  subj=[1, 2, 3, 10, 12]  macro-F1=0.9610

C12_combo_A 5-fold CV macro-F1: 0.9183 ± 0.0419


## In-the-wild robustness - winner only

Reported, not used for selection (same protocol as notebook 13).

In [9]:
G = 9.80665
labels_df = pd.read_csv('../../data/in_the_wild/labels.csv').set_index('session_dir')


def load_session(session_dir):
    base = Path('../../data') / session_dir
    df_ori  = pd.read_csv(base/'Orientation.csv').sort_values('time')
    df_grav = pd.read_csv(base/'Gravity.csv').sort_values('time')
    df_gyr  = pd.read_csv(base/'Gyroscope.csv').sort_values('time')
    df_tot  = pd.read_csv(base/'TotalAcceleration.csv').sort_values('time')
    df = pd.merge_asof(df_ori[['time','roll','pitch','yaw']],
                       df_grav[['time','x','y','z']], on='time', suffixes=('','_grav'))
    df = pd.merge_asof(df, df_gyr[['time','x','y','z']], on='time', suffixes=('','_gyro'))
    df = pd.merge_asof(df, df_tot[['time','x','y','z']], on='time', suffixes=('','_tot_acc'))
    df.columns = ['time','attitude.roll','attitude.pitch','attitude.yaw',
                  'raw_gravity.x','raw_gravity.y','raw_gravity.z',
                  'rotationRate.x','rotationRate.y','rotationRate.z',
                  'raw_total_acc.x','raw_total_acc.y','raw_total_acc.z']
    df['time_dt'] = pd.to_datetime(df['time'])
    df = df.set_index('time_dt').resample('20ms').mean(numeric_only=True).interpolate(method='linear').reset_index(drop=True)
    df['raw_linear_acc.x'] = df['raw_total_acc.x'] - df['raw_gravity.x']
    df['raw_linear_acc.y'] = df['raw_total_acc.y'] - df['raw_gravity.y']
    df['raw_linear_acc.z'] = df['raw_total_acc.z'] - df['raw_gravity.z']
    df['gravity.x'] = -df['raw_gravity.x'] / G
    df['gravity.y'] = -df['raw_gravity.y'] / G
    df['gravity.z'] = -df['raw_gravity.z'] / G
    df['userAcceleration.x'] = -df['raw_linear_acc.x'] / G
    df['userAcceleration.y'] = -df['raw_linear_acc.y'] / G
    df['userAcceleration.z'] = -df['raw_linear_acc.z'] / G
    df['attitude.pitch'] = -df['attitude.pitch']
    df['attitude.yaw']   = -df['attitude.yaw']
    df['attitude.yaw']   = df['attitude.yaw'] - df['attitude.yaw'].iloc[0]
    cols = ['attitude.roll','attitude.pitch','attitude.yaw',
            'gravity.x','gravity.y','gravity.z',
            'rotationRate.x','rotationRate.y','rotationRate.z',
            'userAcceleration.x','userAcceleration.y','userAcceleration.z']
    return df[cols].iloc[150:-150].reset_index(drop=True)


def window_into_batches(arr, w=128, s=64):
    if len(arr) < w: return np.empty((0, w, arr.shape[1]))
    return np.stack([arr[st:st+w] for st in range(0, len(arr)-w+1, s)], axis=0)


winner_model = top3_models[winner_name]['model']

rows = []
for session, row in labels_df.iterrows():
    df_raw = load_session(session)
    feats = compute_walking_frame_features_v2(df_raw, fs_hz=50.0, smooth_seconds=5.0,
                                              group_cols=None, keep_meta=False)
    W = window_into_batches(feats[WALKING_FRAME_V2_COLS].to_numpy())
    if len(W) == 0: continue
    Wn = (W - W.mean(axis=1, keepdims=True)) / (W.std(axis=1, keepdims=True) + 1e-8)
    preds = winner_model.predict(Wn.astype(np.float32), verbose=0).argmax(axis=1)
    correct = float((preds == int(row['activity_id'])).mean())
    majority = int(np.bincount(preds, minlength=6).argmax())
    rows.append({
        'session': session, 'orientation': row['pocket_orientation'],
        'true': row['activity'], 'correct_frac': correct,
        'majority': ACT_LABELS[majority], 'n_windows': len(preds),
    })
    print(f'{session:<22s} gt={row["activity"]:<4s}  correct={correct*100:5.1f}%  majority={ACT_LABELS[majority]}')

itw_df = pd.DataFrame(rows).set_index('session')
weights = itw_df['n_windows'].to_numpy(); fracs = itw_df['correct_frac'].to_numpy()
win_acc  = float(np.average(fracs, weights=weights))
sess_acc = float((fracs > 0.5).mean())
print(f'\n{winner_name} in-the-wild: window-acc={win_acc:.4f}, session-acc={sess_acc:.4f}')
itw_df.to_csv('../results/hp_sweep_baseline_in_the_wild.csv')

dws                    gt=dws   correct=100.0%  majority=dws
ups                    gt=ups   correct=100.0%  majority=ups
hod                    gt=wlk   correct=  0.0%  majority=ups
hod2                   gt=wlk   correct=100.0%  majority=wlk
hodanje                gt=wlk   correct= 61.5%  majority=wlk
hodanje2               gt=wlk   correct=  0.0%  majority=ups
hodanje3               gt=wlk   correct= 62.9%  majority=wlk
jog                    gt=jog   correct=100.0%  majority=jog
ED                     gt=wlk   correct=100.0%  majority=wlk
EG                     gt=wlk   correct= 79.1%  majority=wlk
KD                     gt=wlk   correct= 80.6%  majority=wlk
KG                     gt=wlk   correct= 84.6%  majority=wlk

C12_combo_A in-the-wild: window-acc=0.7417, session-acc=0.8333


## Save final winner

Artifacts are namespaced `cnn_hp_baseline_<winner>.{keras,preproc.json}`
so they don't collide with notebook 13's `cnn_hp_<winner>.*` outputs.

In [10]:
out_path = f'../../models/cnn_hp_baseline_{winner_name}.keras'
winner_model.save(out_path)
with open(out_path.replace('.keras', '.preproc.json'), 'w') as f:
    json.dump({
        'variant': f'A_baseline__{winner_name}',
        'lr': lr_w, 'bs': int(bs_w), 'l2': l2_w,
        'spatial_dropout': sd_w, 'dense_dropout': dd_w, 'label_smoothing': ls_w,
        'channel_order': WALKING_FRAME_V2_COLS,
        'window_size': 128, 'step': 64, 'fs_hz': 50.0, 'smooth_seconds': 5.0,
        'feature_module': 'utils.orientation_invariant_features.compute_walking_frame_features_v2',
        'all_dynamic_zscore': True,
    }, f, indent=2)
print(f'saved {out_path}')
print(f'\nFinal summary:')
print(f'  val macro-F1:       {top3_models[winner_name]["val_f1"]:.4f}')
print(f'  test macro-F1:      {test_df.loc[winner_name, "test_f1"]:.4f}')
print(f'  5-fold CV macro-F1: {fold_f1s.mean():.4f} ± {fold_f1s.std():.4f}')
print(f'  Android window-acc: {win_acc:.4f}')
print(f'  Android session-acc:{sess_acc:.4f}')

saved ../../models/cnn_hp_baseline_C12_combo_A.keras

Final summary:
  val macro-F1:       0.8624
  test macro-F1:      0.9413
  5-fold CV macro-F1: 0.9183 ± 0.0419
  Android window-acc: 0.7417
  Android session-acc:0.8333


## Comparison vs notebook 13 (D_dilated_reg sweep)

Loads both sweep result CSVs and prints a side-by-side summary so
the final architectural decision can be made on a level playing
field (same selection protocol, same compute budget, same data).

In [12]:
summary_rows = [{
    'arch': 'A_baseline',
    'winner': winner_name,
    'val_f1':  top3_models[winner_name]['val_f1'],
    'test_f1': test_df.loc[winner_name, 'test_f1'],
    'cv_mean': fold_f1s.mean(),
    'cv_std':  fold_f1s.std(),
    'itw_win_acc':  win_acc,
    'itw_sess_acc': sess_acc,
}]

d_test_path = Path('../results/hp_sweep_test.csv')
d_val_path  = Path('../results/hp_sweep_val.csv')
if d_test_path.exists() and d_val_path.exists():
    d_test = pd.read_csv(d_test_path).set_index('name')
    d_val  = pd.read_csv(d_val_path).set_index('name')
    d_winner = d_test['test_f1'].idxmax()
    summary_rows.append({
        'arch': 'D_dilated_reg',
        'winner': d_winner,
        'val_f1':  float(d_val.loc[d_winner, 'val_f1']),
        'test_f1': float(d_test.loc[d_winner, 'test_f1']),
        'cv_mean': 0.9248,
        'cv_std':  0.0333,
        'itw_win_acc':  0.7292,
        'itw_sess_acc': 0.8333,
    })
else:
    print('Notebook 13 results not found - run it first to populate the comparison.')

comp_df = pd.DataFrame(summary_rows).set_index('arch')
print(comp_df.round(4).to_string())
comp_df.to_csv('../results/hp_sweep_arch_comparison.csv')

                    winner  val_f1  test_f1  cv_mean  cv_std  itw_win_acc  itw_sess_acc
arch                                                                                   
A_baseline     C12_combo_A  0.8624   0.9413   0.9183  0.0419       0.7417        0.8333
D_dilated_reg   C8_more_sd  0.8418   0.9408   0.9248  0.0333       0.7292        0.8333
